# Modify and write runoff maps
This notebook modifies two existing runoff maps so that they don't put runoff into a set of ocean grid cells where sea ice tends to pile up. The original Matlab implementation is a lot faster, but at least this one is 'free.'

In [1]:
# Import libraries
import numpy as np
import os
import xarray as xr
from netCDF4 import Dataset
from scipy import sparse
import time

In [2]:
# coords = np.array([
#     [27, 183], [32, 138], [33, 138], [33, 140], [33, 141], [37, 202], [39, 135],
#     [39, 204], [39, 205], [45, 391], [50, 136], [51, 278], [56, 139], [56, 258],
#     [57, 140], [58, 240], [62, 320], [62, 321], [62, 322], [63, 321], [63, 322],
#     [77, 489], [78, 489], [78, 490], [86, 503], [87, 503], [87, 504], [88, 506],
#     [89, 25], [438, 397], [458, 383]
# ]) - 1
coords = np.array([[29, 373], [29, 375], [32, 137], [32, 138], [32, 139],
                   [49, 267], [49, 268], [50, 266], [50, 267], [50, 268],
                   [50, 273], [50, 277], [51, 277], [52, 277], [57, 306],
                   [57, 308], [61, 319], [61, 320], [62, 318], [62, 319],
                   [62, 320], [63, 319], [63, 320], [64, 319], [64, 320],
                   [77,   2], [77, 485], [77, 487], [77, 488], [77, 489],
                   [78, 329]])
coords.shape

(31, 2)

In [3]:
def process_runoff(nc_path, target_coords_to_remove):
    # Load NetCDF data (subtract 1 for 0-based indexing)
    with Dataset(nc_path, 'r') as nc:
        row = nc.variables['row'][:] - 1
        col = nc.variables['col'][:] - 1
        s_vals = nc.variables['S'][:]
        area_a = nc.variables['area_a'][:]
        area_b = nc.variables['area_b'][:]

    # Matrix dimensions (540 * 480 = 259200)
    N = 540 * 480
    # Create CSR sparse matrix
    S = sparse.csr_matrix((s_vals, (row, col)), shape=(N, N))
    
    # Use LIL format for efficient row modification
    S = S.tolil()
    
    j_list = []
    # Loop through bad coordinates:
    #  find source indices that map to bad target points
    #  zero out runoff to bad target points
    for coord in target_coords_to_remove:
        # Get target row index for S matrix by converting ni,nj target grid index into linear index, using Fortran ordering
        target_row_idx = np.ravel_multi_index((int(coord[1]), int(coord[0])), (540, 480), order='F')
        
        # Find column indices of nonzero entries in this row.
        # These correspond to source grid cells that runoff into the bad target cells
        col_indices = S.getrow(target_row_idx).nonzero()[1]
        j_list.extend(col_indices)
        
        # Set row to zero. Prevents runoff from entering this target grid cell.
        S[target_row_idx, :] = 0

    # It's possible that some source points map into multiple 'bad' target cells, and their
    # column indices are duplicated in the list. Remove duplicates.
    j_list = np.unique(j_list)
    
    # Conversion for efficient column operations
    S = S.tocsc()
    
    # Loop through source cells that map to bad target cells.
    # If a source point had one (or more) of its target cells zeroed out,
    # increase runoff into the remaining target cells to ensure conservation.
    for idx in j_list:
        col_data = S[:, idx].toarray().flatten()
        S[:, idx] *= area_a[idx] / np.dot(col_data, area_b)

    # Get non-zero elements for saving (Add 1 back to match Fortran 1-based expectations)
    r_final, c_final = S.nonzero()
    s_final = np.array(S[r_final, c_final]).flatten()
    
    return (r_final+1, c_final+1, s_final)

In [4]:
def write_modified_map(original_nc_path, new_nc_path, target_coords_to_remove):
    print('Modifying the map...')
    r_modified, c_modified, s_modified = process_runoff(original_nc_path, target_coords_to_remove)
    print('Writing the modified map...')
    ds_runoff_map = xr.open_dataset(original_nc_path)
    with Dataset(new_nc_path, "w", format="NETCDF4") as rootgrp:
        rootgrp.title = "runoff map: r05 -> tx2_3v3, nearest neighbor and smoothed, target cells removed"
        rootgrp.author = 'Ian Grooms (ian.grooms@colorado.edu)'
        rootgrp.history= "File created " + time.ctime(time.time()) + " using Modify_runoff_maps.ipynb"
        rootgrp.conventions = "NCAR-CCSM"
        rootgrp.domain_a = "/glade/p/cesm/cseg/inputdata/lnd/clm2/rtmdata/rdirc.05.061026"
        rootgrp.domain_b = "/glade/work/gmarques/cesm/tx2_3/mesh/tx2_3v3_260305_SCRIP.nc"
        rootgrp.createDimension('n_a', 259200)
        rootgrp.createDimension('n_b', 259200)
        rootgrp.createDimension('ni_a', 720)
        rootgrp.createDimension('ni_b', 540)
        rootgrp.createDimension('nj_a', 360)
        rootgrp.createDimension('nj_b', 480)
        rootgrp.createDimension('nv_a', 4)
        rootgrp.createDimension('nv_b', 4)
        rootgrp.createDimension('src_grid_rank', 2)
        rootgrp.createDimension('dst_grid_rank', 2)
        rootgrp.createDimension('n_s', s_modified.shape[0])
        rootgrp.createVariable('xc_a','f8',('n_a',),fill_value=np.nan)
        rootgrp["xc_a"].units = "degrees east"
        rootgrp["xc_a"].long_name = "longitude of grid cell center (input)"
        rootgrp["xc_a"][:] = ds_runoff_map.xc_a.data[:]
        rootgrp.createVariable('yc_a','f8',('n_a',),fill_value=np.nan)
        rootgrp["yc_a"].units = "degrees north"
        rootgrp["yc_a"].long_name = "latitude of grid cell center (input)"
        rootgrp["yc_a"][:] = ds_runoff_map.yc_a.data[:]
        rootgrp.createVariable('xv_a','f8',('n_a','nv_a'),fill_value=np.nan)
        rootgrp["xv_a"].units = "degrees east"
        rootgrp["xv_a"].long_name = "longitude of grid cell vertices (input)"
        rootgrp["xv_a"][:] = ds_runoff_map.xv_a.data[:,:]
        rootgrp.createVariable('yv_a','f8',('n_a','nv_a'),fill_value=np.nan)
        rootgrp["yv_a"].units = "degrees north"
        rootgrp["yv_a"].long_name = "latitude of grid cell vertices (input)"
        rootgrp["yv_a"][:] = ds_runoff_map.yv_a.data[:,:]
        rootgrp.createVariable('mask_a','i4',('n_a',))
        rootgrp["mask_a"].long_name = "domain mask (input)"
        rootgrp["mask_a"][:] = ds_runoff_map.mask_a.data[:]
        rootgrp.createVariable('area_a','f8',('n_a',),fill_value=np.nan)
        rootgrp["area_a"].long_name = "area of cell (input)"
        rootgrp["area_a"][:] = ds_runoff_map.area_a.data[:]
        rootgrp.createVariable('frac_a','f8',('n_a',),fill_value=np.nan)
        rootgrp["frac_a"].long_name = "fraction of domain intersection (input)"
        rootgrp["frac_a"][:] = ds_runoff_map.frac_a.data[:]
        rootgrp.createVariable('src_grid_dims','i4',('src_grid_rank',))
        rootgrp["src_grid_dims"][:] = ds_runoff_map.src_grid_dims.data[:]
        
        rootgrp.createVariable('xc_b','f8',('n_b',),fill_value=np.nan)
        rootgrp["xc_b"].units = "degrees east"
        rootgrp["xc_b"].long_name = "longitude of grid cell center (output)"
        rootgrp["xc_b"][:] = ds_runoff_map.xc_b.data[:]
        rootgrp.createVariable('yc_b','f8',('n_b',),fill_value=np.nan)
        rootgrp["yc_b"].units = "degrees north"
        rootgrp["yc_b"].long_name = "latitude of grid cell center (output)"
        rootgrp["yc_b"][:] = ds_runoff_map.yc_b.data[:]
        rootgrp.createVariable('xv_b','f8',('n_b','nv_b'),fill_value=np.nan)
        rootgrp["xv_b"].units = "degrees east"
        rootgrp["xv_b"].long_name = "longitude of grid cell vertices (output)"
        rootgrp["xv_b"][:] = ds_runoff_map.xv_b.data[:,:]
        rootgrp.createVariable('yv_b','f8',('n_b','nv_b'),fill_value=np.nan)
        rootgrp["yv_b"].units = "degrees north"
        rootgrp["yv_b"].long_name = "latitude of grid cell vertices (output)"
        rootgrp["yv_b"][:] = ds_runoff_map.yv_b.data[:,:]
        rootgrp.createVariable('mask_b','i4',('n_b',))
        rootgrp["mask_b"].long_name = "domain mask (output)"
        rootgrp["mask_b"][:] = ds_runoff_map.mask_b.data[:]
        rootgrp.createVariable('area_b','f8',('n_b',),fill_value=np.nan)
        rootgrp["area_b"].long_name = "area of cell (output)"
        rootgrp["area_b"][:] = ds_runoff_map.area_b.data[:]
        rootgrp.createVariable('frac_b','f8',('n_b',),fill_value=np.nan)
        rootgrp["frac_b"].long_name = "fraction of domain intersection (output)"
        rootgrp["frac_b"][:] = ds_runoff_map.frac_b.data[:]
        rootgrp.createVariable('dst_grid_dims','i4',('dst_grid_rank',))
        rootgrp["dst_grid_dims"][:] = ds_runoff_map.dst_grid_dims.data[:]
    
        rootgrp.createVariable('S','f8',('n_s',), fill_value=np.nan)
        rootgrp["S"].long_name = "sparse matrix for mapping S:a->b"
        rootgrp["S"][:] = s_modified
        rootgrp.createVariable('row','i4',('n_s',))
        rootgrp["row"].long_name = "row corresponding to matrix elements"
        rootgrp.createVariable('col','i4',('n_s',))
        rootgrp["row"][:] = r_modified
        rootgrp["col"].long_name = "column corresponding to matrix elements"
        rootgrp["col"][:] = c_modified
    print('Done')

In [6]:
# Modify the merged map with 250 km radius in the NH and 100 km radius in the SH
original_nc_path = '/glade/u/home/igrooms/runoff_mapping/map_r05_to_tx2_3v3_nnsm_e100r100sh_e250r250nh_merged_260307.nc'
new_nc_path = '/glade/u/home/igrooms/runoff_mapping/map_r05_to_tx2_3v3_nnsm_e100r100sh_e250r250nh_merged_modified_260317.nc'
write_modified_map(original_nc_path, new_nc_path, coords)

# # New 100km map:
# original_nc_path = '/glade/work/gmarques/cesm/tx2_3/runoff_mapping/map_r05_to_tx2_3v3_nnsm_e100r100_250327.nc'
# new_nc_path = '/glade/u/home/igrooms/runoff_mapping/map_r05_to_tx2_3v3_nnsm_e100r100_modified_260301.nc'
# write_modified_map(original_nc_path, new_nc_path, coords)
# # New 250km map:
# original_nc_path = '/glade/work/gmarques/cesm/tx2_3/runoff_mapping/map_r05_to_tx2_3v3_nnsm_e250r250_230914.nc'
# new_nc_path = '/glade/u/home/igrooms/runoff_mapping/map_r05_to_tx2_3v3_nnsm_e250r250_modified_260301.nc'
# write_modified_map(original_nc_path, new_nc_path, coords)

Modifying the map...
Writing the modified map...
Done
